# Neutral-atom quantum computing for multi-hypothesis tracking

Run every cell top to bottom. The notebook detects cells in one image,
associates them with tracks, and plots the result.

- A small, quantum-friendly synthetic sequence is used by default. Set
  `USE_QUANTUM_DEMO_DATA = False` to prefer an installed real sequence.
- The preset has eight low-noise frames and a strict eight-node solver cap;
  the default three-frame run includes a real non-clique simulation.
- Switch to the quantum solver in the configuration cell
  (requires `python -m pip install -e ".[quantum]"`).
- Set `RUN_MANY_FRAMES = True` in the last cell to track over a sequence.

In [ ]:
# Imports: make the local src-layout package available before importing it.
from pathlib import Path
import sys

import matplotlib.pyplot as plt

project_root = Path.cwd().resolve()
source_root = project_root / "src"
if not (project_root / "pyproject.toml").is_file() or not source_root.is_dir():
    raise RuntimeError("Open user_notebook.ipynb from the repository root.")
if str(source_root) not in sys.path:
    sys.path.insert(0, str(source_root))

from neutral_atom_mht import (
    ClassicalSolver,
    DEFAULT_SYNTHETIC_DATA_ROOT,
    HPC,
    HPCConfig,
    QUANTUM_DEMO_DATA_CONFIG,
    QuantumSolver,
    SyntheticDataConfig,
    SyntheticDataGenerator,
    SyntheticDataset,
)
from cell_data import DATASET_NAME, FRAME_COUNT, load_tiff, raw_frame_path

In [ ]:
# Data: use the bounded quantum demo by default. Set the flag to False to
# prefer an installed real sequence, with synthetic data as its fallback.
frame = 0
real_dataset_root = project_root / "data" / DATASET_NAME
USE_QUANTUM_DEMO_DATA = True
USE_SYNTHETIC_DATA = (
    USE_QUANTUM_DEMO_DATA
    or not raw_frame_path(real_dataset_root, frame).is_file()
)

# This versioned eight-frame preset keeps quantum components small while
# retaining one genuine non-clique Pulser simulation.
synthetic_data_config = QUANTUM_DEMO_DATA_CONFIG
synthetic_output_root = project_root / DEFAULT_SYNTHETIC_DATA_ROOT
synthetic_dataset = SyntheticDataset(
    root=synthetic_output_root / synthetic_data_config.dataset_name,
    config=synthetic_data_config,
)
if USE_SYNTHETIC_DATA and not synthetic_dataset.raw_frame_path(0).is_file():
    print(f"Real data not found; generating {synthetic_dataset.root} ...")
    synthetic_dataset = SyntheticDataGenerator(synthetic_data_config).generate(
        synthetic_output_root
    )
    print("Synthetic sequence generated.")

In [ ]:
# Configuration: resolve the selected data source and choose a solver.
if USE_SYNTHETIC_DATA:
    dataset_root = synthetic_dataset.root
    frame_path = synthetic_dataset.raw_frame_path(frame)
    sequence = synthetic_dataset.config.sequence
    dataset_label = synthetic_data_config.dataset_name
    available_frames = synthetic_data_config.frame_count
else:
    dataset_root = real_dataset_root
    frame_path = raw_frame_path(dataset_root, frame)
    sequence = "01"
    dataset_label = DATASET_NAME
    available_frames = FRAME_COUNT
print(f"Using {dataset_label} sequence {sequence}: {frame_path}")

config = HPCConfig()
controller = HPC(config, sequence=sequence)
# The quantum demo preset was validated with components of at most five nodes.
# solver = ClassicalSolver(maximum_component_nodes=60)
# Optional quantum simulation (requires `python -m pip install -e ".[quantum]"`).
# The explicit cap prevents accidental exponential simulation growth.
solver = QuantumSolver(maximum_component_nodes=8)

In [ ]:
# Run: detect the cells, create tracks, and show the detections on the source frame.
image = load_tiff(frame_path)
prepared = controller.prepare_frame(image, frame=frame)
solver_result = controller.solve(prepared, solver)
result = controller.advance(prepared, solver_result)
detections = prepared.observed_frame.detection.detections

figure, axis = plt.subplots(figsize=(10, 8))
axis.imshow(image, cmap="gray")
axis.scatter(
    [detection.x_px for detection in detections],
    [detection.y_px for detection in detections],
    s=28,
    facecolors="none",
    edgecolors="tab:red",
    linewidths=0.9,
    label="detected cell",
)
axis.set_title(
    f"{dataset_label} sequence {sequence}, frame {frame:03d}: "
    f"{len(detections)} detections"
)
axis.set_axis_off()
axis.legend(loc="upper right")
plt.show()

{
    "dataset": dataset_label,
    "frame_path": str(frame_path),
    "detections": len(detections),
    "initialized_tracks": len(result.tracks),
    "solver": solver.solver_name,
}

In [ ]:
# Optional multi-frame run: enable explicitly, especially before using QuantumSolver.
RUN_MANY_FRAMES = True
MANY_FRAME_COUNT = 3
sequence_summary = None
if RUN_MANY_FRAMES:
    frames_to_run = min(MANY_FRAME_COUNT, available_frames)
    if frames_to_run < 1:
        raise ValueError("MANY_FRAME_COUNT must be positive")

    if USE_SYNTHETIC_DATA:
        frame_paths = [
            synthetic_dataset.raw_frame_path(frame_index)
            for frame_index in range(frames_to_run)
        ]
    else:
        frame_paths = [
            raw_frame_path(dataset_root, frame_index)
            for frame_index in range(frames_to_run)
        ]
    images = (load_tiff(path) for path in frame_paths)

    sequence_controller = HPC(config, sequence=sequence)
    sequence_result = sequence_controller.run_sequence(
        images, solver, start_frame=0
    )
    processed_frames = [step.frame for step in sequence_result.steps]
    active_tracks = [len(step.tracks) for step in sequence_result.steps]
    assigned_observations = [
        len(step.assigned_observation_ids) for step in sequence_result.steps
    ]

    figure, axis = plt.subplots(figsize=(10, 4))
    axis.plot(processed_frames, active_tracks, label="active tracks")
    axis.plot(
        processed_frames,
        assigned_observations,
        label="assigned observations",
    )
    axis.set(
        xlabel="frame",
        ylabel="count",
        title=(
            f"{dataset_label}: {frames_to_run} frames with "
            f"{sequence_result.solver_name}"
        ),
    )
    axis.grid(alpha=0.25)
    axis.legend()
    plt.show()

    sequence_summary = {
        "frames_processed": len(sequence_result.steps),
        "final_tracks": len(sequence_result.final_tracks),
        "solver": sequence_result.solver_name,
        "solver_runtime_seconds": sum(
            step.solver_result.runtime_seconds for step in sequence_result.steps
        ),
    }
sequence_summary

## Publication figures

The following cells save six reproducible figures under `outputs/figures/`.
The benchmark cells require the optional quantum dependencies and take about
20 seconds on the validated preset. Figures 3–6 use frame 2 of that preset.
Quantum and exact scores are evaluated on the same immutable frame graph before
the exact-reference controller advances, so their comparison is like-for-like.
Because the tracker retains one state per track rather than a population of
global hypotheses, Fig. 6 reports the cumulative frame-local association score.

In [ ]:
# Shared publication-figure imports, output path, and rendering helpers.
from dataclasses import replace
from itertools import combinations
from time import perf_counter

import numpy as np
from matplotlib.lines import Line2D
from matplotlib.patches import Circle, FancyArrowPatch, FancyBboxPatch

from graph import logical_layout

figure_output_dir = project_root / "outputs" / "figures"
figure_output_dir.mkdir(parents=True, exist_ok=True)

def finish_figure(figure, filename):
    path = figure_output_dir / filename
    figure.tight_layout()
    figure.savefig(path, dpi=180, bbox_inches="tight")
    plt.show()
    plt.close(figure)
    return path

def draw_detection_overlay(axis, image, detection, title):
    axis.imshow(image, cmap="gray")
    if np.any(detection.labels):
        axis.contour(detection.labels > 0, levels=[0.5], colors="#00D4FF", linewidths=0.7)
    axis.scatter(
        [item.x_px for item in detection.detections],
        [item.y_px for item in detection.detections],
        s=42, facecolors="none", edgecolors="#FF4D4D", linewidths=1.2,
    )
    axis.set_title(f"{title} ({len(detection.detections)} detections)")
    axis.set_axis_off()

In [ ]:
# Fig. 1 — End-to-end image, solver, and tracking workflow.
figure, axis = plt.subplots(figsize=(16, 4.2))
axis.set_xlim(0, 1)
axis.set_ylim(0, 1)
axis.axis("off")

workflow_boxes = [
    (0.01, 0.43, 0.11, 0.25, "Image frame", "#D9EAF7"),
    (0.15, 0.43, 0.11, 0.25, "Detect\nobservations", "#D9EAF7"),
    (0.29, 0.43, 0.13, 0.25, "Predict, gate,\nand weight", "#FCE8C3"),
    (0.45, 0.43, 0.12, 0.25, "Weighted\nconflict graph", "#FCE8C3"),
    (0.60, 0.35, 0.14, 0.41, "Solver\n\nClassical exact\nor\nneutral atom:\nembed → pulse → sample", "#E4D7F5"),
    (0.77, 0.43, 0.10, 0.25, "Selected\nassociations", "#D7F0E3"),
    (0.90, 0.43, 0.09, 0.25, "Bayesian +\nKalman update", "#D7F0E3"),
]
for x, y, width, height, label, color in workflow_boxes:
    box = FancyBboxPatch(
        (x, y), width, height, boxstyle="round,pad=0.012",
        facecolor=color, edgecolor="#263238", linewidth=1.4,
    )
    axis.add_patch(box)
    axis.text(x + width / 2, y + height / 2, label, ha="center", va="center", fontsize=10)
for left, right in zip(workflow_boxes, workflow_boxes[1:]):
    start = (left[0] + left[2], left[1] + left[3] / 2)
    stop = (right[0], right[1] + right[3] / 2)
    axis.add_patch(FancyArrowPatch(start, stop, arrowstyle="-|>", mutation_scale=14, color="#455A64"))
axis.add_patch(FancyArrowPatch(
    (0.945, 0.42), (0.355, 0.42), connectionstyle="arc3,rad=-0.35",
    arrowstyle="-|>", mutation_scale=14, color="#00796B", linewidth=1.6,
))
axis.text(0.66, 0.06, "retained tracks become the next frame's prior state", ha="center", color="#00796B")
axis.set_title("Fig. 1 — Neutral-atom multi-hypothesis tracking workflow", fontsize=14, pad=12)
fig1_path = finish_figure(figure, "fig1_workflow.png")
fig1_path

In [ ]:
# Fig. 2 — Real and simulated detection overlays.
synthetic_figure_dataset = synthetic_dataset
if not synthetic_figure_dataset.raw_frame_path(0).is_file():
    synthetic_figure_dataset = SyntheticDataGenerator(synthetic_data_config).generate(
        synthetic_output_root
    )
synthetic_image = synthetic_figure_dataset.load_frame(0)
synthetic_detection = HPC(config, sequence=synthetic_data_config.sequence).observe(
    synthetic_image, frame=0
).detection

figure, axes = plt.subplots(1, 2, figsize=(14, 5.5))
real_path = raw_frame_path(real_dataset_root, 0)
if real_path.is_file():
    real_image = load_tiff(real_path)
    real_detection = HPC(config, sequence="01").observe(real_image, frame=0).detection
    draw_detection_overlay(axes[0], real_image, real_detection, "Real data, frame 000")
else:
    axes[0].set_facecolor("#F3F4F6")
    axes[0].text(0.5, 0.56, "Real sequence not installed", ha="center", va="center", fontsize=13)
    axes[0].text(0.5, 0.44, str(real_path), ha="center", va="center", fontsize=8, color="#666666", wrap=True)
    axes[0].set_title("Real data, frame 000")
    axes[0].set_xticks([])
    axes[0].set_yticks([])
draw_detection_overlay(
    axes[1], synthetic_image, synthetic_detection,
    f"Simulated {synthetic_data_config.dataset_name}, frame 000",
)
figure.suptitle("Fig. 2 — Image detection overlays", fontsize=14)
fig2_path = finish_figure(figure, "fig2_detection_overlays.png")
{"path": fig2_path, "real_data_available": real_path.is_file()}

In [ ]:
# Shared real-simulator benchmark for Figs. 3–6.
# Classical results advance one reference controller; quantum and exact
# are always evaluated first on the same immutable PreparedFrame.
figure_noise_levels = (0.00, 0.05, 0.10, 0.15)
comparison_records = []
example_prepared = None
example_execution = None
example_exact_result = None

for noise_level in figure_noise_levels:
    if np.isclose(noise_level, QUANTUM_DEMO_DATA_CONFIG.noise):
        benchmark_config = QUANTUM_DEMO_DATA_CONFIG
    else:
        noise_code = int(round(1000 * noise_level))
        benchmark_config = replace(
            QUANTUM_DEMO_DATA_CONFIG,
            noise=noise_level,
            dataset_name=f"SYN-MHT-QUANTUM-N{noise_code:03d}-v1",
        )
    benchmark_dataset = SyntheticDataset(
        root=synthetic_output_root / benchmark_config.dataset_name,
        config=benchmark_config,
    )
    complete_cache = (
        benchmark_dataset.track_manifest_path.is_file()
        and all(
            benchmark_dataset.raw_frame_path(index).is_file()
            and benchmark_dataset.tracking_frame_path(index).is_file()
            for index in range(benchmark_config.frame_count)
        )
    )
    if not complete_cache:
        if benchmark_dataset.root.exists():
            raise RuntimeError(
                f"Incomplete cached benchmark dataset: {benchmark_dataset.root}"
            )
        benchmark_dataset = SyntheticDataGenerator(benchmark_config).generate(
            synthetic_output_root
        )

    reference = HPC(HPCConfig(), sequence=benchmark_config.sequence)
    quantum_benchmark = QuantumSolver(maximum_component_nodes=8)
    exact_benchmark = ClassicalSolver(maximum_component_nodes=8)
    for benchmark_frame in range(benchmark_config.frame_count):
        prepared_benchmark = reference.prepare_frame(
            benchmark_dataset.load_frame(benchmark_frame),
            frame=benchmark_frame,
        )
        solver_input = prepared_benchmark.solver_input()
        quantum_started = perf_counter()
        quantum_execution = quantum_benchmark.execute(solver_input)
        quantum_runtime = perf_counter() - quantum_started
        if not quantum_execution.successful:
            message = quantum_execution.diagnostics.get("message", "no details")
            raise RuntimeError(
                f"Quantum benchmark failed for noise={noise_level:.2f}, "
                f"frame={benchmark_frame}: {quantum_execution.status} ({message})"
            )
        exact_result = exact_benchmark.solve(solver_input)
        if not exact_result.successful:
            raise RuntimeError(f"Exact benchmark failed at frame {benchmark_frame}")
        quantum_objective = sum(
            solver_input.graph.node(node_id).weight
            for node_id in quantum_execution.selected_ids
        )
        relative_objective = (
            quantum_objective / exact_result.objective
            if exact_result.objective > 0.0
            else np.nan
        )
        comparison_records.append(
            {
                "noise": noise_level,
                "frame": benchmark_frame,
                "quantum_objective": quantum_objective,
                "exact_objective": exact_result.objective,
                "relative_objective": relative_objective,
                "quantum_status": quantum_execution.status,
                "selection_agrees": (
                    quantum_execution.selected_ids == exact_result.selected_ids
                ),
                "quantum_runtime_seconds": quantum_runtime,
                "exact_runtime_seconds": exact_result.runtime_seconds,
            }
        )
        if np.isclose(noise_level, QUANTUM_DEMO_DATA_CONFIG.noise) and benchmark_frame == 2:
            example_prepared = prepared_benchmark
            example_execution = quantum_execution
            example_exact_result = exact_result
        reference.advance(prepared_benchmark, exact_result)

if example_prepared is None or example_execution is None or not example_execution.successful:
    raise RuntimeError("The validated frame-2 quantum example was not produced")
simulated_example_runs = tuple(
    run for run in example_execution.runs if run.program is not None
)
if not simulated_example_runs:
    raise RuntimeError(
        "Frame 2 contains no simulated component; regenerate the validated preset"
    )
example_run = max(
    simulated_example_runs,
    key=lambda run: (len(run.node_ids), -run.component_id),
)
benchmark_summary = {
    noise: {
        "statuses": sorted({row["quantum_status"] for row in comparison_records if row["noise"] == noise}),
        "quantum_seconds": sum(row["quantum_runtime_seconds"] for row in comparison_records if row["noise"] == noise),
        "minimum_relative_objective": min(
            row["relative_objective"] for row in comparison_records
            if row["noise"] == noise and np.isfinite(row["relative_objective"])
        ),
        "selection_agreement": all(
            row["selection_agrees"] for row in comparison_records
            if row["noise"] == noise
        ),
    }
    for noise in figure_noise_levels
}
benchmark_summary

In [ ]:
# Fig. 3 — Logical embedding of the frame-2 weighted conflict graph.
example_graph = example_prepared.graph
graph_positions = logical_layout(example_graph)
quantum_selected = set(example_execution.selected_ids)
figure, axis = plt.subplots(figsize=(9, 6))
for left, right in example_graph.edges:
    axis.plot(
        [graph_positions[left][0], graph_positions[right][0]],
        [graph_positions[left][1], graph_positions[right][1]],
        color="#8795A1", linewidth=1.8, zorder=1,
    )
for node in example_graph.nodes:
    x_value, y_value = graph_positions[node.node_id]
    selected = node.node_id in quantum_selected
    axis.scatter(
        [x_value], [y_value], s=1100,
        color="#2A9D8F" if selected else "#E9C46A",
        edgecolor="#173F5F", linewidth=2.5 if selected else 1.2, zorder=2,
    )
    axis.text(
        x_value, y_value,
        f"h{node.node_id}\nT{node.track_id}→O{node.observation_id}\nw={node.weight:.2f}",
        ha="center", va="center", fontsize=8, zorder=3,
    )
axis.set_title("Fig. 3 — Frame 002 weighted association-conflict graph")
axis.text(0.5, -0.04, "edge = shared track or observation; teal = quantum-selected hypothesis", transform=axis.transAxes, ha="center")
axis.set_aspect("equal", adjustable="datalim")
axis.axis("off")
fig3_path = finish_figure(figure, "fig3_conflict_graph.png")
fig3_path

In [ ]:
# Fig. 4 — Stylized physical neutral-atom representation of that component.
program = example_run.program
if program is None:
    raise RuntimeError("The selected example has no neutral-atom program")
component = program.component
coordinates = dict(zip(example_run.node_ids, example_run.coordinates, strict=True))
qubit_by_node = dict(zip(component.node_ids, component.qubit_ids, strict=True))
blockade_distance = program.sequence.device.rydberg_blockade_radius(program.omega)
selected_atoms = set(example_run.selected_ids)
intended_edges = {tuple(sorted(edge)) for edge in component.edges}
physical_edges = set()
for left, right in combinations(component.node_ids, 2):
    separation = np.linalg.norm(
        np.asarray(coordinates[left]) - np.asarray(coordinates[right])
    )
    if separation <= blockade_distance:
        physical_edges.add(tuple(sorted((left, right))))

figure, axis = plt.subplots(figsize=(8, 7))
for node_id, (x_value, y_value) in coordinates.items():
    axis.add_patch(Circle(
        (x_value, y_value), blockade_distance / 2.0,
        facecolor="#4CC9F0", edgecolor="#168AAD", alpha=0.08, linewidth=1.0,
    ))
for left, right in sorted(intended_edges | physical_edges):
    if (left, right) in intended_edges and (left, right) in physical_edges:
        color, linestyle, linewidth = "#2A9D8F", "-", 2.3
    elif (left, right) in intended_edges:
        color, linestyle, linewidth = "#D62828", "--", 2.0
    else:
        color, linestyle, linewidth = "#7B2CBF", ":", 2.0
    axis.plot(
        [coordinates[left][0], coordinates[right][0]],
        [coordinates[left][1], coordinates[right][1]],
        color=color, linestyle=linestyle, linewidth=linewidth, zorder=1,
    )
for node_id in component.node_ids:
    x_value, y_value = coordinates[node_id]
    selected = node_id in selected_atoms
    axis.scatter(
        [x_value], [y_value], s=430,
        color="#E63946" if selected else "#F4A261",
        edgecolor="#1D3557", linewidth=1.6, zorder=3,
    )
    axis.text(
        x_value, y_value, f"{qubit_by_node[node_id]}\nh{node_id}",
        ha="center", va="center", fontsize=8, color="white", zorder=4,
    )
legend_handles = [
    Line2D([0], [0], marker="o", color="none", markerfacecolor="#E63946", markeredgecolor="#1D3557", markersize=11, label="selected Rydberg atom"),
    Line2D([0], [0], marker="o", color="none", markerfacecolor="#F4A261", markeredgecolor="#1D3557", markersize=11, label="unselected atom"),
    Line2D([0], [0], marker="o", color="none", markerfacecolor="#4CC9F0", alpha=0.25, markersize=14, label="$R_b/2$ interaction halo"),
]
if intended_edges & physical_edges:
    legend_handles.append(Line2D([0], [0], color="#2A9D8F", linewidth=2.3, label="intended edge realized by blockade"))
if intended_edges - physical_edges:
    legend_handles.append(Line2D([0], [0], color="#D62828", linestyle="--", linewidth=2.0, label="intended edge outside blockade"))
if physical_edges - intended_edges:
    legend_handles.append(Line2D([0], [0], color="#7B2CBF", linestyle=":", linewidth=2.0, label="unintended blockade pair"))
axis.legend(handles=legend_handles, loc="upper right")
realized_count = len(intended_edges & physical_edges)
axis.set(
    xlabel="x coordinate (μm)", ylabel="y coordinate (μm)",
    title=("Fig. 4 — Heuristic neutral-atom embedding of the simulated component\n"
           f"mapping cost={example_run.mapping_cost:.3f}; "
           f"realized intended edges={realized_count}/{len(intended_edges)}; "
           f"selected hypotheses={example_run.selected_ids}"),
)
axis.set_aspect("equal", adjustable="datalim")
axis.grid(alpha=0.15)
fig4_path = finish_figure(figure, "fig4_neutral_atom_embedding.png")
fig4_path

In [ ]:
# Fig. 5 — Quantum objective relative to exact MWIS under increasing noise.
figure, axis = plt.subplots(figsize=(10, 5.5))
color_map = plt.get_cmap("viridis")
normalization = plt.Normalize(min(figure_noise_levels), max(figure_noise_levels))
markers = ("o", "s", "^", "D")
frame_offsets = np.linspace(-0.09, 0.09, len(figure_noise_levels))
for marker, noise_level, frame_offset in zip(
    markers, figure_noise_levels, frame_offsets, strict=True
):
    rows = sorted(
        (row for row in comparison_records if row["noise"] == noise_level),
        key=lambda row: row["frame"],
    )
    axis.plot(
        [row["frame"] + frame_offset for row in rows],
        [row["relative_objective"] for row in rows],
        marker=marker, markerfacecolor="white", markeredgewidth=1.6,
        linewidth=1.2, markersize=6, alpha=0.9,
        color=color_map(normalization(noise_level)), label=f"noise={noise_level:.2f}",
    )
axis.axhline(1.0, color="#333333", linestyle=":", linewidth=1.2, label="exact objective")
axis.set(
    xlabel="frame number",
    ylabel="quantum objective / exact objective",
    ylim=(0.985, 1.005),
    title="Fig. 5 — Quantum performance on exact-reference frame graphs",
)
axis.set_xticks(range(QUANTUM_DEMO_DATA_CONFIG.frame_count))
axis.grid(alpha=0.25)
axis.legend(ncol=3, loc="lower left")
axis.text(
    0.99, 0.90,
    "frame 0 omitted (0/0); markers offset ±0.09 frame to reveal coincident curves",
    transform=axis.transAxes, ha="right", fontsize=8, color="#444444",
    bbox={"facecolor": "white", "edgecolor": "none", "alpha": 0.8},
)
color_scale = plt.cm.ScalarMappable(norm=normalization, cmap=color_map)
figure.colorbar(color_scale, ax=axis, label="synthetic noise/difficulty level")
fig5_path = finish_figure(figure, "fig5_performance_vs_exact.png")
fig5_path

In [ ]:
# Fig. 6 — Cumulative frame-local objective on the exact-reference trajectory.
# Each MWIS vertex weight is the association log-score gain over assigning a
# missed detection. The tracker does not retain a population of global hypotheses.
default_rows = sorted(
    (row for row in comparison_records if np.isclose(row["noise"], QUANTUM_DEMO_DATA_CONFIG.noise)),
    key=lambda row: row["frame"],
)
frames = np.asarray([row["frame"] for row in default_rows])
quantum_cumulative_objective = np.cumsum(
    [row["quantum_objective"] for row in default_rows]
)
exact_cumulative_objective = np.cumsum(
    [row["exact_objective"] for row in default_rows]
)
figure, axis = plt.subplots(figsize=(10, 5.5))
axis.plot(frames, exact_cumulative_objective, color="#264653", linewidth=3.0, label="classical exact")
axis.plot(frames, quantum_cumulative_objective, color="#E76F51", linestyle="--", marker="o", linewidth=2.0, label="neutral-atom quantum")
axis.set(
    xlabel="frame number",
    ylabel="cumulative MWIS gain over all-missed baseline (log-score units)",
    title="Fig. 6 — Cumulative frame-local association objective",
)
axis.grid(alpha=0.25)
axis.legend()
fig6_path = finish_figure(figure, "fig6_cumulative_association_score.png")
fig6_path